# Quantum Feature Map for Track Geometry Prediction

**Research question:** Does encoding track geometry parameters into a quantum feature space — via a ZZ Feature Map capturing nonlinear and periodic cross-parameter interactions — yield better one-month-ahead predictions than classical models?

This notebook covers:
1. Data loading and inspection (prototype: 2 months)
2. Exploratory data analysis
3. Feature engineering (windowed inputs + first differences)
4. *(upcoming)* Quantum circuit design and feature extraction
5. *(upcoming)* Classical vs. quantum model comparison

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
DATA_PATH = Path('Data/Raw Data.xlsx')

## 1. Data Loading

The dataset contains monthly track geometry inspections (June 2013 – April 2016, 28 months total).
We prototype with **two consecutive months**: June 2013 and July 2013.

Each row is a track measurement point (~5,270 points per month). Key geometry columns:
| Column | Description |
|---|---|
| `GAGE` | Gauge (distance between rails) |
| `XLEVEL` | Crosslevel |
| `WARP_62` | Warp (62-ft baseline) |
| `PROFSC_L` / `PROFSC_R` | Profile score — left / right rail |
| `ALIGNSC_L` / `ALIGNSC_R` | Alignment score — left / right rail |

In [ ]:
# Geometry feature columns used throughout
GEO_COLS = ['GAGE', 'XLEVEL', 'WARP_62', 'PROFSC_L', 'PROFSC_R', 'ALIGNSC_L', 'ALIGNSC_R']
ID_COLS   = ['MILE', 'FEET', 'TRACK', 'MP']

# Prototype months — change these to explore other pairs
MONTHS = ['6_2013', '7_2013']

xl = pd.ExcelFile(DATA_PATH)
print('All available months:', [s for s in xl.sheet_names if s != 'Chart'])

In [ ]:
def load_month(sheet_name: str) -> pd.DataFrame:
    """Load a single monthly sheet and return with a parsed date column."""
    df = xl.parse(sheet_name)
    df['month_label'] = sheet_name
    return df

frames = {m: load_month(m) for m in MONTHS}

for m, df in frames.items():
    print(f'{m}: {df.shape[0]} rows, {df.shape[1]} columns')

In [ ]:
# Quick look at one month
frames[MONTHS[0]][ID_COLS + GEO_COLS].head()

In [ ]:
# Summary statistics for geometry columns — both months
for m, df in frames.items():
    print(f'\n=== {m} ===')
    display(df[GEO_COLS].describe().round(4))

## 2. Exploratory Data Analysis

In [ ]:
# Distribution of each geometry parameter across both months
fig, axes = plt.subplots(len(GEO_COLS), 1, figsize=(10, 3 * len(GEO_COLS)), sharex=False)

colors = ['steelblue', 'tomato']
for ax, col in zip(axes, GEO_COLS):
    for (m, df), c in zip(frames.items(), colors):
        ax.hist(df[col].dropna(), bins=80, alpha=0.6, color=c, label=m, density=True)
    ax.set_title(col)
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Geometry Parameter Distributions — Month Comparison', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap per month
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
import matplotlib.colors as mcolors

for ax, (m, df) in zip(axes, frames.items()):
    corr = df[GEO_COLS].corr()
    im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(GEO_COLS)))
    ax.set_yticks(range(len(GEO_COLS)))
    ax.set_xticklabels(GEO_COLS, rotation=45, ha='right')
    ax.set_yticklabels(GEO_COLS)
    ax.set_title(f'Correlation — {m}')
    for i in range(len(GEO_COLS)):
        for j in range(len(GEO_COLS)):
            ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
# Spatial profile along track milepost for one parameter
fig, axes = plt.subplots(len(GEO_COLS), 1, figsize=(12, 3 * len(GEO_COLS)), sharex=True)

for ax, col in zip(axes, GEO_COLS):
    for (m, df), c in zip(frames.items(), colors):
        ax.plot(df['MP'].values, df[col].values, '.', markersize=1, alpha=0.4, color=c, label=m)
    ax.set_ylabel(col)
    ax.legend(loc='upper right', markerscale=5)

axes[-1].set_xlabel('Milepost (MP)')
plt.suptitle('Geometry Parameters Along Track — Month Comparison', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3. Feature Engineering

For the prediction task we align measurement points across months by milepost (`MP`), then construct:
- **Current values** at time *t*: the 7 geometry columns
- **First differences** (rate of change): Δ = value(t) − value(t−1)
- **Target**: geometry values at *t+1* (next month)

Scaling to [0, π] is applied for quantum angle encoding (fitted on training data only).

In [ ]:
def align_months(df_t0: pd.DataFrame, df_t1: pd.DataFrame, cols: list) -> pd.DataFrame:
    """
    Align two monthly DataFrames on milepost (MP) and construct
    windowed features: current values + first differences + next-month target.
    """
    # Round MP to avoid floating-point mismatch
    df_t0 = df_t0.copy()
    df_t1 = df_t1.copy()
    df_t0['MP_round'] = df_t0['MP'].round(4)
    df_t1['MP_round'] = df_t1['MP'].round(4)

    merged = df_t0[['MP_round'] + cols].merge(
        df_t1[['MP_round'] + cols],
        on='MP_round',
        suffixes=('_t0', '_t1')
    ).dropna()

    # First differences
    for c in cols:
        merged[f'd_{c}'] = merged[f'{c}_t1'] - merged[f'{c}_t0']

    # Feature matrix (t0 values + first differences) and targets (t1 values)
    feature_cols = [f'{c}_t0' for c in cols] + [f'd_{c}' for c in cols]
    target_cols  = [f'{c}_t1' for c in cols]

    X = merged[feature_cols].copy()
    y = merged[target_cols].copy()
    X.columns = [c.replace('_t0', '') for c in feature_cols]
    y.columns = cols

    return X, y, merged['MP_round']


X, y, mp = align_months(frames[MONTHS[0]], frames[MONTHS[1]], GEO_COLS)
print(f'Aligned samples: {len(X)}')
print(f'Feature columns ({len(X.columns)}): {list(X.columns)}')
print(f'Target columns  ({len(y.columns)}): {list(y.columns)}')
X.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Time-based split (respect temporal order — no shuffle)
split = int(0.8 * len(X))
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f'Train: {len(X_train)} | Test: {len(X_test)}')

# Scale features to [0, π] for quantum angle encoding
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Target scaler (for reporting predictions in original units)
target_scaler = MinMaxScaler()
y_train_scaled = target_scaler.fit_transform(y_train)
y_test_scaled  = target_scaler.transform(y_test)

print('Feature range after scaling (should be [0, π]):')
print(f'  min={X_train_scaled.min():.4f}  max={X_train_scaled.max():.4f}')

## 4. Next Steps

- [ ] **ZZ Feature Map implementation** — encode scaled features as quantum expectation values (Tier 1: ⟨Zᵢ⟩, Tier 2: ⟨ZᵢZⱼ⟩, Tier 3: ⟨ZᵢZⱼZₖ⟩)
- [ ] **Classical baselines** — C1 (raw), C2 (polynomial), C3 (trigonometric)
- [ ] **Model training** — Ridge, XGBoost, CatBoost on each feature configuration
- [ ] **Evaluation** — RMSE, MAE, R² + Wilcoxon significance tests
- [ ] **Interpretability** — SHAP, circuit ablation, Fourier spectrum analysis
- [ ] **Scale up** — extend from 2 months to full 28-month dataset